In [1]:
import pprint
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

import os

from dotenv import load_dotenv

load_dotenv("C:\\Users\\socgen\\ML\\agentic_ai_and_ops\\langchain_day5\\.env")

True

In [2]:
model_gr_lamma = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)


In [6]:
from pydantic import BaseModel, Field
from typing import Literal, Union

from langchain.agents import create_agent

from langchain.agents.structured_output import ToolStrategy

In [7]:

class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")


In [9]:

agent = create_agent(
    model=model_gr_lamma,
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # Default: handle_errors=True
)


In [11]:

result=agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

In [12]:
from pprint import pprint

pprint(result, depth=4, compact=True)

{'messages': [HumanMessage(content='Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th', additional_kwargs={}, response_metadata={}, id='68f00559-5946-45b3-a78c-14488537f8a5'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3351an92d', 'function': {'arguments': '{"email":"john@email.com","name":"John Doe"}', 'name': 'ContactInfo'}, 'type': 'function'}, {'id': 'mfngh4rpr', 'function': {'arguments': '{"date":"March 15th","event_name":"Tech Conference"}', 'name': 'EventDetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 311, 'total_tokens': 357, 'completion_time': 0.095823499, 'completion_tokens_details': None, 'prompt_time': 0.015710303, 'prompt_tokens_details': None, 'queue_time': 0.052969367, 'total_time': 0.111533802}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logpro

In [ ]:
tool_messages = [
    msg
    for msg in result["messages"]
    if getattr(msg, "type", None) == "tool" or msg.__class__.__name__ == "ToolMessage"
]

for tool_msg in tool_messages:
    pprint(tool_msg, depth=4, compact=True)

ProductReview(product_name='Mobile', review_text='Great Mobile: 5 out of 5 stars. Fast shipping, but expensive', rating=5, shipping_sentiment='positive', pricing_sentiment='negative', quality_sentiment='positive')

In [17]:
result["messages"][-1]

ToolMessage(content='Please provide the product review analysis in the specified format.', name='ProductReview', id='5ed55057-a2a1-43e7-b279-22d54d8201c8', tool_call_id='bbr592bwv')

In [53]:
result["structured_response"].model_dump_json()

'{"product_name":"Mobile","review_text":"Great Mobile: 5 out of 5 stars. Fast shipping, but expensive","rating":5,"shipping_sentiment":"positive","pricing_sentiment":"negative","quality_sentiment":"positive"}'